In [ ]:
# ============================================================
# BIASCOPE
# Multilingual & Code-Mixed News Bias Detection
#
# Stage 1: English
# Stage 2: Indic Languages
# Stage 3: Hinglish
#
# Models:
#   - RoBERTa / XLM-R / MuRIL
#   - LoRA / QLoRA
#
# Analysis:
#   - Bias Detection
#   - Evidence Extraction
#   - LLM-based Explanation
# ============================================================

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("⚠️ GPU not available. Enable GPU from Runtime → Change runtime type.")

In [ ]:
!pip install -q "pyarrow>=19,<24"

In [ ]:
!pip install -q -U \
    "torchao>=0.16.0" \
    peft \
    transformers \
    accelerate \
    datasets \
    evaluate \
    bitsandbytes \
    sentencepiece \
    scikit-learn

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

In [ ]:
PROJECT_NAME = "BIASCOPE"

LANGUAGES = [
    "english",
    "hindi",
    "hinglish"
]

TASK = "bias_detection"

LABELS = {
    0: "neutral",
    1: "biased"
}

NUM_LABELS = len(LABELS)

print("Project:", PROJECT_NAME)
print("Task:", TASK)
print("Languages:", LANGUAGES)
print("Labels:", LABELS)

In [ ]:
BASE_DIR = Path("/content/biascope")

directories = [
    "data/raw",
    "data/processed",
    "data/english",
    "data/hindi",
    "data/hinglish",
    "models",
    "outputs",
    "logs",
    "experiments"
]

for directory in directories:
    (BASE_DIR / directory).mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)

for directory in directories:
    print("✓", directory)

In [ ]:
!find /content/biascope -maxdepth 3 -type d

In [ ]:
ENGLISH_MODEL = "roberta-base"

MAX_LENGTH = 256

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

print("English model:", ENGLISH_MODEL)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)

In [ ]:
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1

In [ ]:
EXPERIMENT = {
    "project": PROJECT_NAME,
    "task": TASK,
    "language": "english",
    "model": ENGLISH_MODEL,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "seed": SEED
}

print(json.dumps(EXPERIMENT, indent=4))

In [ ]:
if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {total_memory:.2f} GB")
    print(f"Allocated: {allocated_memory:.2f} GB")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENGLISH_MODEL)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
sample_text = """
The government announced a new economic policy yesterday.
"""

tokens = tokenizer(
    sample_text,
    truncation=True,
    max_length=MAX_LENGTH
)

print(tokens.keys())
print("Input IDs:", tokens["input_ids"][:20])
print("Number of tokens:", len(tokens["input_ids"]))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    ENGLISH_MODEL,
    num_labels=NUM_LABELS
)

print("Model loaded successfully.")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none"
)

lora_model = get_peft_model(
    model,
    lora_config
)

lora_model.print_trainable_parameters()